In [1]:
import pandas as pd
import numpy as np

In [ ]:
# import sys
# print(sys.executable)

/usr/local/bin/python3


In [ ]:
# import sys
# !{sys.executable} -m pip install feedparser
# !{sys.executable} -m pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 22.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip available: 22.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import feedparser
import duckdb
from datetime import datetime
import re

In [13]:
# ---------------------------
# CONFIG
# ---------------------------

RSS_FEEDS = {
    # "ge": "https://ge.globo.com/rss/ge/futebol/",
    "ge": "http://globoesporte.globo.com/ESP/Noticia/Rss/0,,AS0-4271,00.xml",
    "espn": "https://www.espn.com.br/rss/futebol",
    "uol": "https://www.uol.com.br/rss.xml",
    "terra": "https://www.terra.com.br/rss/feeds/futebol.xml",
    "yahoo": "https://br.sports.yahoo.com/rss/"
}

TIMES = {
    "São Paulo": ["são paulo", "tricolor paulista", "spfc"],
    "Palmeiras": ["palmeiras", "verdão"],
    "Corinthians": ["corinthians", "timão"],
    "Santos": ["santos", "peixe"],
    "Flamengo": ["flamengo", "mengão"]
}

DB_PATH = "noticias_futebol.duckdb"

# ---------------------------
# FUNÇÕES
# ---------------------------

def identificar_time(texto):
    texto = texto.lower()
    for time, palavras in TIMES.items():
        for p in palavras:
            if re.search(rf"\b{p}\b", texto):
                return time
    return None


def coletar_noticias():
    registros = []

    for portal, url in RSS_FEEDS.items():
        feed = feedparser.parse(url)

        for entry in feed.entries:
            titulo = entry.get("title", "")
            resumo = entry.get("summary", "")
            link = entry.get("link", "")
            publicado = entry.get("published", None)

            texto_base = f"{titulo} {resumo}"
            time = identificar_time(texto_base)

            if time:
                registros.append({
                    "portal": portal,
                    "time": time,
                    "titulo": titulo,
                    "resumo": resumo,
                    "link": link,
                    "data_publicacao": publicado,
                    "data_coleta": datetime.utcnow()
                })

    return pd.DataFrame(registros)


def salvar_duckdb(df):
    if df.empty:
        print("Nenhuma notícia encontrada.")
        return

    con = duckdb.connect(DB_PATH)
    con.execute("""
        CREATE TABLE IF NOT EXISTS noticias (
            portal TEXT,
            time TEXT,
            titulo TEXT,
            resumo TEXT,
            link TEXT,
            data_publicacao TEXT,
            data_coleta TIMESTAMP
        )
    """)
    con.register("df", df)
    con.execute("INSERT INTO noticias SELECT * FROM df")
    con.close()


# ---------------------------
# MAIN
# ---------------------------

# if __name__ == "__main__":
#     df_noticias = coletar_noticias()
#     salvar_duckdb(df_noticias)
#     print(f"{len(df_noticias)} notícias salvas com sucesso.")

In [11]:
df_noticias = coletar_noticias()
salvar_duckdb(df_noticias)
print(f"{len(df_noticias)} notícias salvas com sucesso.")

Nenhuma notícia encontrada.
0 notícias salvas com sucesso.


In [6]:
import requests
from bs4 import BeautifulSoup

URL = "https://ge.globo.com/futebol/times/palmeiras/"

def extrair_noticias(url=URL):
    resp = requests.get(url, headers={
        "User-Agent": "Mozilla/5.0"
    })
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    noticias = []

    # Ajuste os seletores conforme a estrutura atual da página:
    # geralmente as manchetes ficam em <a> ou <h2>/<h3> com classes específicas.
    for bloco in soup.select("a"):  # comece genérico, depois refine o seletor
        titulo = bloco.get_text(strip=True)
        link = bloco.get("href")

        # filtro simples pra evitar lixo
        if not titulo or not link:
            continue
        if "palmeiras" not in link and "/futebol/times/palmeiras" not in link and "verdao" not in link:
            continue

        # links relativos -> absolutos
        if link.startswith("/"):
            link = "https://ge.globo.com" + link

        noticias.append({
            "titulo": titulo,
            "link": link
        })

    return noticias

if __name__ == "__main__":
    noticias = extrair_noticias()
    for n in noticias:
        print(n["titulo"])
        print(n["link"])
        print("-" * 80)


palmeiras
http://globoesporte.globo.com/futebol/times/palmeiras/
--------------------------------------------------------------------------------
últimas
https://ge.globo.com/futebol/times/palmeiras/
--------------------------------------------------------------------------------
jogos do palmeiras
https://ge.globo.com/futebol/times/palmeiras/agenda-de-jogos-do-palmeiras/
--------------------------------------------------------------------------------
vídeos
https://ge.globo.com/futebol/times/palmeiras/videos/
--------------------------------------------------------------------------------
Tudo sobre o Palmeiras na Globo, sportv, ge e ge tv
https://ge.globo.com/futebol/times/palmeiras/playlist/videos-do-palmeiras-no-sportv.ghtml
--------------------------------------------------------------------------------
Palmeiras empresta Raphael Veiga a time mexicano
https://ge.globo.com/futebol/times/palmeiras/noticia/2026/01/30/palmeiras-empresta-raphael-veiga-ao-america-do-mexico.ghtml
-------

In [ ]:
ge = "http://globoesporte.globo.com/ESP/Noticia/Rss/0,,AS0-4271,00.xml"
df_noticias = coletar_noticias()

In [3]:
import feedparser

url = "https://news.google.com/..."  # seu link
feed = feedparser.parse(url)
print(feed.bozo, feed.bozo_exception)


1 <unknown>:2:0: syntax error


In [4]:
import feedparser
from datetime import datetime
import csv

RSS_URL = ("https://news.google.com/rss/topics/"
           "CAAqJggKIiBDQkFTRWdvSkwyMHZNREU1YkhkaUVnVndkQzFDVWlnQVAB"
           "?hl=pt-BR&gl=BR&ceid=BR:pt-419&oc=11")

def parse_google_news_rss(url=RSS_URL):
    feed = feedparser.parse(url)

    noticias = []
    for entry in feed.entries:
        titulo = entry.title

        # Em Google News RSS, a fonte costuma vir em entry.source.title
        portal = getattr(entry, "source", {}).get("title", "") if hasattr(entry, "source") else ""

        link = entry.link

        # Alguns itens têm published/published_parsed, outros updated/updated_parsed
        if hasattr(entry, "published_parsed"):
            dt_struct = entry.published_parsed
        elif hasattr(entry, "updated_parsed"):
            dt_struct = entry.updated_parsed
        else:
            dt_struct = None

        if dt_struct:
            data_hora = datetime(*dt_struct[:6])  # ano, mes, dia, hora, min, seg
            data_hora_str = data_hora.strftime("%Y-%m-%d %H:%M:%S")
        else:
            data_hora_str = ""

        noticias.append({
            "titulo": titulo,
            "portal": portal,
            "link": link,
            "data_hora_publicacao": data_hora_str,
        })

    return noticias

def salvar_em_csv(noticias, caminho_arquivo="noticias_google_news.csv"):
    campos = ["titulo", "portal", "link", "data_hora_publicacao"]
    with open(caminho_arquivo, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        writer.writeheader()
        writer.writerows(noticias)

if __name__ == "__main__":
    noticias = parse_google_news_rss()
    salvar_em_csv(noticias)

    # Exemplo de impressão
    for n in noticias[:5]:
        print(n["data_hora_publicacao"], "-", n["portal"], "-", n["titulo"])
        print(n["link"])
        print("-" * 80)


In [18]:
feed = feedparser.parse("https://news.google.com/rss/topics/"
           "CAAqJggKIiBDQkFTRWdvSkwyMHZNREU1YkhkaUVnVndkQzFDVWlnQVAB"
           "?hl=pt-BR&gl=BR&ceid=BR:pt-419&oc=11")

In [19]:
feed

{'bozo': True,
 'entries': [],
 'feed': {},
 'headers': {},
 'bozo_exception': urllib.error.URLError(ssl.SSLCertVerificationError(1,
                                                    '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:992)'))}

In [5]:
noticias = parse_google_news_rss("https://news.google.com/rss/topics/"
           "CAAqJggKIiBDQkFTRWdvSkwyMHZNREU1YkhkaUVnVndkQzFDVWlnQVAB"
           "?hl=pt-BR&gl=BR&ceid=BR:pt-419&oc=11")
noticias

[]